### Without SemanticSimilarityExampleSelector
### FewShotChatMessagePromptTemplate

In [1]:
print("Hi there!")

Hi there!


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage,AIMessage
from PromptUtils import get_llm 
llm = get_llm()

In [3]:
history = []

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")])

In [5]:
from langchain_core.output_parsers import StrOutputParser
chain = prompt | llm | StrOutputParser()

In [23]:
input_text = input("Enter Query: ")
response = chain.invoke({'input': input_text, 'history': history})
history.append(HumanMessage(content=input_text))
history.append(AIMessage(content=response)) 
print(response)
history=history[-4:]

I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. Therefore, I don't know your name. If you'd like to share it or if you have any other questions, feel free to let me know!


In [24]:
for i in history:
    print(i)

content='who r u' additional_kwargs={} response_metadata={}
content="I am an AI language model created by OpenAI, designed to assist with a variety of questions and tasks. I'm here to provide information, answer questions, and help you with anything you need. How can I assist you today?" additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]
content='wha is my name' additional_kwargs={} response_metadata={}
content="I don't have access to personal data about individuals unless it has been shared with me in the course of our conversation. Therefore, I don't know your name. If you'd like to share it or if you have any other questions, feel free to let me know!" additional_kwargs={} response_metadata={} tool_calls=[] invalid_tool_calls=[]


# Structure Output

In [39]:
from langchain_core.prompts import PromptTemplate

In [40]:
inputs = [
    "mohan aget 45 software engineer located in Hyderabad",
    "sita aget 25 bank enployee "
]

In [41]:
prompt_template = PromptTemplate.from_template("Extract the name, age, occupation and location from the following text: {text}")

In [42]:
chian = prompt_template | llm | StrOutputParser()
response = chian.invoke(inputs[0])
print(response)


Name: Mohan  
Age: 45  
Occupation: Software Engineer  
Location: Hyderabad


In [43]:
prompt_json = PromptTemplate.from_template("Extract the name, age, occupation and location from the following text: {text}" \
"Make sure you must return the output in JSON format with keys name, age, occupation and location" \
"If any of the information is missing, return null for that key.")
chian = prompt_json | llm | StrOutputParser()
response = chian.invoke(inputs[1])
print(response)

```json
{
  "name": "sita",
  "age": 25,
  "occupation": "bank employee",
  "location": null
}
```


# Structure Pydantic model
### Using PydanticOutputParser

In [59]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Optional

class PersonInfo(BaseModel):
    name: str = Field(..., description="The person's name")
    age: int = Field(..., description="The person's age")
    occupation: str = Field(..., description="The person's occupation")
    location: Optional[str] = Field(None, description="The person's location") 

prompt_pydantic = PromptTemplate.from_template("Extract the name, age, occupation and location from the following text: {text}" \
"If any of the information is missing, return null for that key.")
parser = PydanticOutputParser(pydantic_object=PersonInfo)
chian = prompt_pydantic | llm | parser
response = chian.invoke(inputs[0])
print(response)

name='mohan' age=45 occupation='software engineer' location='Hyderabad'


### Using with structured output llm

In [58]:
struc_llm=llm.with_structured_output(PersonInfo)
st_chain = prompt_pydantic | struc_llm 
response = st_chain.invoke(inputs[0])
print(response)
type(response.age)


name='mohan' age=45 occupation='software engineer' location='Hyderabad'


int